In [1]:
import numpy as np

In [2]:
import cupy as cp

In [3]:
!nvidia-smi

Fri Jul 10 15:59:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10                     On  |   00000000:01:00.0 Off |                    0 |
|  0%   59C    P0             92W /  150W |    2575MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
pip install cupy-cuda13x

Note: you may need to restart the kernel to use updated packages.


In [5]:
import cupy as cp

In [6]:
pip install cuda-cccl[cu13]

Note: you may need to restart the kernel to use updated packages.


# `cuda.compute`: Parallel Computing Primitives

The `cuda.compute` library provides composable primitives for building custom parallel algorithms on the GPU—without writing CUDA kernels directly.

https://nvidia.github.io/cccl/unstable/python/compute/index.html



In [7]:
"""
Sum all values in an array using reduction with PLUS operation.
"""

import cupy as cp
import numpy as np

import cuda.compute
from cuda.compute import OpKind

# Prepare the input and output arrays.
dtype = np.int32
h_init = np.array([0], dtype=dtype)
d_input = cp.array([1, 2, 3, 4, 5], dtype=dtype)
d_output = cp.empty(1, dtype=dtype)

# Perform the reduction.
cuda.compute.reduce_into(
    d_in=d_input, d_out=d_output, num_items=len(d_input), op=OpKind.PLUS, h_init=h_init
)

# Verify the result.
expected_output = 15
assert (d_output == expected_output).all()
result = d_output[0]
print(f"Sum reduction result: {result}")

Sum reduction result: 15


## Stream-Based Computing with `cuda.compute` Iterators

When designing high-performance parallel pipelines on the GPU, memory bandwidth is almost always the primary bottleneck. Traditional array processing frameworks (like standard NumPy or naive CuPy workflows) operate by executing one operation at a time across the entire dataset, allocating temporary intermediate arrays in video memory (VRAM) along the way.

For instance, if you want to take an array, square every element, and then sum them up, a traditional pipeline allocates an entirely new scratchpad array just to hold the squared values before passing it to the reduction engine. When dealing with millions of elements, this constant allocating, writing, and reading from VRAM degrades execution speeds.

To solve this, `cuda.compute` introduces a lazy, stream-based abstraction model: **Iterators**.

```
--- TRADITIONAL VECTOR OPERATIONS (Allocates Intermediate VRAM Scratchpads) ---
[Input Array] ──(Read/Write)──> [Temporary Array (x^2)] ──(Read/Write)──> [Reduction Engine]

--- STREAM-BASED ITERATOR FUSION (Zero Extra Allocations) ---
[Input Array] ──(Stream Elements On-The-Fly via Iterator)──> [Reduction Engine]

```

Iterators do not allocate memory buffers. Instead, they represent a recipe or a continuous mathematical stream of data. They allow you to **fuse transformations and structure data layouts directly inside the execution pipeline**, piping elements straight into parallel primitives (like `reduce_into` or `scan_into`) on the fly.

---

### The 4 Core Iterator Primitives

The library provides four foundational iterators that can be combined to structure incoming parallel data streams:

#### 1. TransformIterator

A `TransformIterator` applies a unary function (a transformation lambda or a JIT-compiled function) to every element of an underlying input stream as it is requested by the algorithm.

* **Physics Application:** Useful for lazy coordinate or unit conversions (e.g., converting an array of track momentum values from $\text{GeV}$ to $\text{MeV}$ inside the histogramming step without generating a secondary modified array).
* **Syntax Blueprint:** `TransformIterator(d_input_array, lambda x: x * 1000.0)`

#### 2. CountingIterator

A `CountingIterator` generates a deterministic, infinite sequential sequence of numbers starting from a defined initialization point. It consumes zero memory because values are computed arithmetically on the fly by each GPU thread based on its absolute index position.

* **Physics Application:** Generating track indices, grid coordinates, or matching time-bin identifiers.
* **Syntax Blueprint:** `CountingIterator(np.int32(0))` (Note: The starting point value must be typed explicitly as a host NumPy array or scalar object).

#### 3. ZipIterator

A `ZipIterator` pairs two or more independent data streams together horizontally, yielding a single stream of structured tuples or records. Each parallel thread reads from the corresponding index across all zipped iterators simultaneously.

* **Physics Application:** Organizing a Structure of Arrays (SoA) layout. For example, if you have separate, contiguous device arrays for `d_px`, `d_py`, and `d_pz`, you can zip them together so that your downstream optimization kernel receives them as a cohesive 3D momentum vector `(px, py, pz)` per thread.
* **Syntax Blueprint:** `ZipIterator(it_stream_A, it_stream_B)`

#### 4. DiscardIterator

A `DiscardIterator` acts as an analytical garbage disposal or sink buffer. It is an output iterator that accepts values from an algorithm but completely discards them instead of writing them to memory.

* **Physics Application:** Streamlining multi-output algorithms where you only care about a specific subset of metrics (e.g., executing a complex prefix scan or key-sorting operation where you want to retain the indices or keys but want to drop the transformed values entirely without paying a VRAM write penalty).
* **Syntax Blueprint:** Used as a target destination placeholder for `d_out`.

## Composable GPU Primitives with `cuda.compute`
This interactive tutorial introduces `cuda.compute`, an executive library providing high-performance, parallel computing primitives on the GPU without requiring manual CUDA kernel development. Under the hood, these primitives leverage Numba CUDA to just-in-time (JIT) compile expressively written Python functions straight into accelerated device code.

## Core Architecture & Conventions

Before tackling the puzzles, remember these three strict API design constraints required by `cuda.compute`:  

 - Keyword-Only Arguments: Every algorithm parameter must be explicitly passed by name. Positional arguments will immediately trigger a TypeError.
    - ❌ Incorrect: `cuda.compute.reduce_into(d_in, d_out, n, OpKind.PLUS, h_init)`
    -    Correct: `cuda.compute.reduce_into(d_in=d_in, d_out=d_out, num_items=n, op=OpKind.PLUS, h_init=h_init)`
 - Hungarian Memory Prefixing: Buffer naming explicitly tracks physical layout:
    - `d_` prefixes identify variables residing in Device memory (e.g., CuPy arrays).
    - `h_` prefixes identify variables residing in Host memory (e.g., NumPy scalar containers).
 - Explicit Output Semantics: Primitives write directly into pre-allocated output target destinations rather than returning newly allocated objects. This provides precise control over memory lifecycles on the GPU.

### Puzzle 1: Fused Operation Streaming (5-Minute Estimate)
#### Objective
Given an array containing integers, compute the sum of squares of only the odd values in a single pass across the data without allocating intermediate arrays for the transformations.

#### 💡 Hint
You can fuse transformations directly into the reduction pipeline using a `TransformIterator`. It lazily applies a unary lambda operation on demand as data passes into the reduction framework.

In [8]:
import cupy as cp
import numpy as np
import cuda.compute
from cuda.compute import OpKind, TransformIterator

# Target data sequence
h_data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=np.int32)
d_input = cp.array(h_data)

# Pre-allocated single-element destination buffer on device
d_output = cp.empty(1, dtype=np.int32)

# Host initialization container for the accumulator
h_init = np.array([0], dtype=np.int32)

#### Code Challenge
Complete the implementation below to construct a lazy mapping stream and reduce it into `d_output`:

In [9]:
# 1. Instantiate a TransformIterator to lazily evaluate: 
#    x**2 if x is odd, else 0
it_input = TransformIterator(
    d_input, 
    lambda x: # ### YOUR CODE HERE ###
)

# 2. Execute the reduction using explicit keyword-only arguments
cuda.compute.reduce_into(
    # ### YOUR CODE HERE ###
)

# Validate results
expected_sum = sum(x**2 for x in h_data if x % 2 != 0)
assert int(d_output[0]) == expected_sum
print(f"✅ Success! Fused sum of odd squares: {int(d_output[0])}")

SyntaxError: invalid syntax (3718027637.py, line 6)

#### Solution Spoiler

In [10]:
it_input = TransformIterator(d_input, lambda x: x**2 if x % 2 != 0 else 0)

cuda.compute.reduce_into(
    d_in=it_input,
    d_out=d_output,
    num_items=len(d_input),
    op=OpKind.PLUS,
    h_init=h_init
)

In [11]:
d_output

array([165], dtype=int32)

## Puzzle 2: Parallel Argmin Lookup via Tuples (10-Minute Estimate)
### Objective
Locate both the **minimum value** and its associated **index** inside an array using a single parallel passes across a Structure of Arrays (SoA) layout.  
### 💡 Hint
You can group disjoint parallel streams together horizontally using a `ZipIterator`. When pairing a sequential `CountingIterator` with an array, the zipper streams logical structural tuple elements (`(index, value)`) down to your customized device reduction function.  Because the reduction handles structured pairs, the initialization state (`h_init`) and target buffer (`d_output`) must match a matching structured data type. 

In [12]:
import cupy as cp
import numpy as np
import cuda.compute
from cuda.compute import CountingIterator, ZipIterator

# Target dataset array containing an explicit global minimum value
h_dataset = np.array([42, 17, 89, 5, 66, 12, 91, 4], dtype=np.int32)
d_dataset = cp.array(h_dataset)

# Configure the tracking structured data type
pair_dtype = np.dtype([("index", np.int32), ("value", np.int32)], align=True)

# Allocate host and device metadata tracking boundaries
h_init = np.asarray([(-1, np.iinfo(np.int32).max)], dtype=pair_dtype)
d_output = cp.empty(1, dtype=pair_dtype)

#### Code Challenge
Write the custom binary comparison operation and compile the stream zipper sequence:

In [13]:
# 1. Define the JIT-compiled reduction logic tracking the absolute minimum value
def min_by_value(p1, p2):
    """
    Evaluates two structured tuple records and passes onwards 
    the one containing the smaller value element.
    """
    # ### YOUR CODE HERE ###


# 2. Instantiate a sequential coordinate position counter starting at index 0
counting_it = # ### YOUR CODE HERE ###


# 3. Zip coordinate positions together with physical target metrics
zip_it = # ### YOUR CODE HERE ###


# 4. Invoke the reduction pipeline using mandatory keyword configurations
cuda.compute.reduce_into(
    # ### YOUR CODE HERE ###
)

# Validate correctness
result = d_output.get()[0]
expected_idx = int(np.argmin(h_dataset))
expected_val = int(h_dataset[expected_idx])

assert result["index"] == expected_idx and result["value"] == expected_val
print(f"✅ Success! Global Argmin Located at Index [{result['index']}] -> Value: {result['value']}")

SyntaxError: invalid syntax (1275974516.py, line 11)

#### Solution Spoiler

In [14]:
def min_by_value(p1, p2):
    return p1 if p1[1] < p2[1] else p2

counting_it = CountingIterator(np.int32(0))
zip_it = ZipIterator(counting_it, d_dataset)

cuda.compute.reduce_into(
    d_in=zip_it,
    d_out=d_output,
    num_items=len(d_dataset),
    op=min_by_value,
    h_init=h_init
)

In [15]:
d_output

array([(7, 4)],
      dtype={'names': ['index', 'value'], 'formats': ['<i4', '<i4'], 'offsets': [0, 4], 'itemsize': 8, 'aligned': True})